# UpsureAI — Car Damage Classifier Training
**Run on Google Colab with T4 GPU (~20-25 min for ~3520 images)**

## Before running:
1. **Runtime → Change runtime type → T4 GPU** (do this first)
2. Create a folder in Google Drive: `MyDrive/UpsureAI/`
3. Upload these 2 files to that folder:
   - `images.zip` — zip of local `images/` folder (all 3519 car images)
   - `dataset.csv` — already built (3520 rows, 969 cars)
4. Run all cells top to bottom

## How to create the zip (run locally in PowerShell):
```powershell
Compress-Archive -Path C:\Users\sriva\Desktop\UpsureAI\images\* -DestinationPath C:\Users\sriva\Desktop\UpsureAI\images.zip
```

In [ ]:
# ── CONFIGURE THIS ─────────────────────────────────────────────────────────────
DRIVE_FOLDER = 'MyDrive/UpSureAI'          # your Drive folder path
ZIP_IMAGES   = 'images.zip'                 # all 3519 images
CSV_NAME     = 'dataset.csv'                # 3520 rows, 969 cars
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install timm albumentations==1.4.3 scikit-learn -q

In [ ]:
import os, shutil, zipfile
from pathlib import Path

DRIVE_BASE = Path(f'/content/drive/{DRIVE_FOLDER}')
WORK_DIR   = Path('/content/upsureai')
IMG_DIR    = WORK_DIR / 'images'

WORK_DIR.mkdir(exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)

def extract_zip_flat(zip_path: Path, dest: Path):
    with zipfile.ZipFile(zip_path) as z:
        entries = z.namelist()
        skipped = 0
        extracted = 0
        for name in entries:
            fname = Path(name).name
            if not fname or not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp')):
                continue
            target = dest / fname
            if target.exists():
                skipped += 1
                continue
            with z.open(name) as src, open(target, 'wb') as dst:
                dst.write(src.read())
            extracted += 1
        print(f'  {zip_path.name}: {extracted} extracted, {skipped} skipped')

print('Extracting images...')
zp = DRIVE_BASE / ZIP_IMAGES
if zp.exists():
    extract_zip_flat(zp, IMG_DIR)
else:
    print(f'  ERROR: {ZIP_IMAGES} not found in Drive at {DRIVE_BASE}')

print(f'Total images in {IMG_DIR}: {len(list(IMG_DIR.iterdir()))}')

In [ ]:
import csv

CSV_PATH  = DRIVE_BASE / CSV_NAME
LOCAL_CSV = WORK_DIR / CSV_NAME
shutil.copy(CSV_PATH, LOCAL_CSV)

rows = list(csv.DictReader(open(LOCAL_CSV, encoding='utf-8')))

for row in rows:
    fname = Path(row['image_path'].replace('\\', '/')).name
    row['image_path'] = str(IMG_DIR / fname)

fixed_csv = WORK_DIR / 'dataset_colab.csv'
with open(fixed_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

missing = [r for r in rows if not Path(r['image_path']).exists()]
present = len(rows) - len(missing)
print(f'Total rows : {len(rows)}')
print(f'Found      : {present}')
print(f'Missing    : {len(missing)}')
if missing:
    print('Sample missing:', [Path(r["image_path"]).name for r in missing[:5]])

labels_all = [int(r['label']) for r in rows if Path(r['image_path']).exists()]
print(f'Damaged    : {sum(labels_all)}  ({100*sum(labels_all)/len(labels_all):.1f}%)')
print(f'Clean      : {len(labels_all)-sum(labels_all)}  ({100*(1-sum(labels_all)/len(labels_all)):.1f}%)')

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from PIL import Image
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# ── Config ─────────────────────────────────────────────────────────────────────
MODEL_NAME   = 'efficientnet_b2'
IMG_SIZE     = 260
BATCH_SIZE   = 32
EPOCHS       = 25
LR           = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 2
SEED         = 42
MEAN         = [0.485, 0.456, 0.406]
STD          = [0.229, 0.224, 0.225]
OUT_PTH      = '/content/upsureai/damage_model.pth'
OUT_ONNX     = '/content/upsureai/damage_model.onnx'
CSV_FILE     = str(fixed_csv)
# ───────────────────────────────────────────────────────────────────────────────

torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


class CarDamageDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records   = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row   = self.records[idx]
        img   = np.array(Image.open(row['image_path']).convert('RGB'))
        label = int(row['label'])
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label


def get_transforms(train):
    if train:
        return A.Compose([
            A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0)),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
            A.HueSaturationValue(p=0.4),
            A.GaussianBlur(blur_limit=(3, 5), p=0.2),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(height=IMG_SIZE, width=IMG_SIZE),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        probs  = torch.softmax(logits, 1)[:, 1]
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    return total_loss / total, correct / total, auc, all_probs, all_labels


records = [r for r in csv.DictReader(open(CSV_FILE, encoding='utf-8'))
           if Path(r['image_path']).exists()]
print(f'Training on {len(records)} images')

labels_arr = [int(r['label']) for r in records]
train_rec, temp_rec = train_test_split(records, test_size=0.3, stratify=labels_arr, random_state=SEED)
temp_labels = [int(r['label']) for r in temp_rec]
val_rec, test_rec = train_test_split(temp_rec, test_size=0.5, stratify=temp_labels, random_state=SEED)
print(f'Train: {len(train_rec)}  Val: {len(val_rec)}  Test: {len(test_rec)}')

train_ds = CarDamageDataset(train_rec, get_transforms(True))
val_ds   = CarDamageDataset(val_rec,   get_transforms(False))
test_ds  = CarDamageDataset(test_rec,  get_transforms(False))

train_labels  = [int(r['label']) for r in train_rec]
class_counts  = [train_labels.count(0), train_labels.count(1)]
weights       = [1.0 / class_counts[l] for l in train_labels]
sampler       = WeightedRandomSampler(weights, len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,    num_workers=NUM_WORKERS, pin_memory=True)

model     = timm.create_model(MODEL_NAME, pretrained=True, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_auc = 0.0
print(f'\nTraining {MODEL_NAME} for {EPOCHS} epochs on {device}\n')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc, vl_auc, _, _ = eval_epoch(model, val_loader, criterion)
    scheduler.step()
    elapsed = time.time() - t0
    print(f'Epoch {epoch:>3}/{EPOCHS}  '
          f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f}  '
          f'val_loss={vl_loss:.4f} val_acc={vl_acc:.3f} val_auc={vl_auc:.3f}  '
          f'[{elapsed:.1f}s]')
    if vl_auc > best_auc:
        best_auc = vl_auc
        torch.save(model.state_dict(), OUT_PTH)
        print(f'  New best AUC={best_auc:.4f}')

In [ ]:
model.load_state_dict(torch.load(OUT_PTH, map_location=device))
_, te_acc, te_auc, probs, true_labels = eval_epoch(model, test_loader, criterion)
preds = [1 if p >= 0.25 else 0 for p in probs]

print('TEST RESULTS')
print(f'Accuracy : {te_acc:.4f}')
print(f'AUC-ROC  : {te_auc:.4f}')
print(classification_report(true_labels, preds, target_names=['Clean', 'Damaged']))

In [ ]:
!pip install onnxscript -q

In [ ]:
model.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
torch.onnx.export(
    model, dummy, OUT_ONNX,
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)
print(f'ONNX saved: {OUT_ONNX}')

In [ ]:
import shutil
drive_out = DRIVE_BASE / 'model_output'
drive_out.mkdir(parents=True, exist_ok=True)

shutil.copy(OUT_PTH,  drive_out / 'damage_model.pth')
shutil.copy(OUT_ONNX, drive_out / 'damage_model.onnx')
print(f'Models saved to Drive: {drive_out}')

from google.colab import files
files.download(OUT_PTH)
files.download(OUT_ONNX)
print('Download started.')
print('Save both files to: C:\\Users\\sriva\\Desktop\\UpsureAI\\models\\')